# 02 — Factor returns and specific returns

> **Universe build** (`ryan/universe`). Same model code as `ryan/sample`; the
> universe, tables and history come from `config.py` — profile set by the
> `BARRA_UNIVERSE` environment variable (default `r1000`). See the README.

**Step 4 of the model.** One constrained, cap-weighted cross-sectional regression
per month:

$$r_i(t) = f_{\text{market}}(t) + \sum_p X_{ip}(t-1)\, f_p^{\text{ind}}(t) + \sum_k X_{ik}(t-1)\, f_k^{\text{sty}}(t) + u_i(t)$$

The fitted coefficients are the **factor returns** `f(t)`; the residuals are the
**specific returns** `u(i,t)`. They are the same object seen from two sides and
cannot be computed separately — a specific return is *defined* as what the factor
returns fail to explain.

Reads `ryan_<profile>_exposures`. Writes `ryan_<profile>_factor_returns`,
`ryan_<profile>_specific_returns` and `ryan_<profile>_regression_diag`. Run top to
bottom.

### Three things here are not a plain `lstsq` call

1. **The design matrix is rank-deficient by construction.** The 11 industry
   dummies sum, row by row, to the market factor's column of ones. `lstsq` returns
   *a* minimum-norm solution, but it is an arbitrary point on a one-dimensional
   family of equally good fits and it drifts month to month for numerical reasons
   alone — which would make `F` garbage. USE4 pins it with the cap-weighted
   constraint $\sum_p \text{CAP}_p f_p^{\text{ind}} = 0$.

2. **Exposures are lagged one month.** `ret_exc` at eom *t* is the return
   *during* month *t*, while `market_equity` at eom *t* is the cap *after* that
   month's move. Since `be_me`, `ni_me`, `ocf_me` and log size all divide by (or
   are) that market equity, a contemporaneous fit hands the regression the very
   return it is meant to explain. Cell 6 measures the damage.

3. **The dependent variable is winsorized** (not standardized — it keeps its
   units), as secondary insurance against extreme returns. Stage 5 measures how
   much of the work each fix actually does; the answer is not the obvious one.

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

# ryan/universe/config.py -- imported FIRST: it shadows the sample's config for
# every shared module and puts ryan/sample (where the model code lives) on the path.
import config
assert config.PROFILE != "sample", "wrong config on the path — run this notebook from ryan/universe"
print(config.describe())

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import db
import factor_returns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# Validated categorical palette: passes the lightness band, chroma floor,
# adjacent-pair CVD separation and the normal-vision floor. The three lower-
# contrast hues are always paired with a direct label, never colour alone.
STYLE_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
INK, ACCENT, MUTED = "#2a78d6", "#e34948", "#8a8a83"

print(f"lag exposures         {config.LAG_EXPOSURES}")
print(f"return winsor sigma   {config.WINSOR_RETURN_SIGMA}")
print(f"reading               {config.TBL_EXPOSURES}")

In [ ]:
engine = db.get_engine()
db.check_connection(engine)

---
## Stage 1 — Load the exposures

`X(t)` comes straight out of 01. Nothing is recomputed here: if the exposures
need to change, change them there and re-run that notebook, so the stored `X`
and the `f` fitted from it never disagree.

In [ ]:
if not db.table_exists(engine, config.TBL_EXPOSURES):
    raise RuntimeError(
        f"{config.TBL_EXPOSURES} is missing — run run_pipeline.ipynb first."
    )

exposures = db.read_table(config.TBL_EXPOSURES, engine)
exposures["eom"] = pd.to_datetime(exposures["eom"])

style_cols = [c for c in exposures.columns if c.startswith("f_")]
dummy_cols = [c for c in exposures.columns if c.startswith("ind_")]
n_factors = 1 + len(dummy_cols) + len(style_cols)

print(f"📂 {len(exposures):,} rows / {exposures['id'].nunique()} securities / "
      f"{exposures['eom'].nunique()} months")
print(f"   {exposures['eom'].min().date()} → {exposures['eom'].max().date()}")
print(f"\nK = {n_factors}: 1 market + {len(dummy_cols)} industries + {len(style_cols)} styles")
print(f"   styles:     {', '.join(c[2:] for c in style_cols)}")
print(f"   industries: {', '.join(dummy_cols)}")
exposures.head()

---
## Stage 2 — The timing problem, measured

Before fitting anything, confirm that the lag is necessary rather than
cosmetic. The test: correlate each exposure with the **same month's** return,
cross-sectionally, then do it again with the exposure lagged one month.

If the panel were properly aligned these two columns would look similar. They do
not. `be_me` uses market equity at eom *t*, so a stock that fell during month *t*
mechanically gets a higher book-to-market — and the regression reads that as
"value stocks underperformed."

In [ ]:
# Mean cross-sectional correlation of two columns, averaged over months.
# Columns are selected before .apply so the grouping key is not passed into the
# lambda -- that is what pandas 2.2 deprecated, and selecting explicitly avoids
# the `include_groups` argument, which does not exist before 2.2.
def xs_corr(a, b, groups, min_n=10):
    d = pd.DataFrame({"a": a, "b": b, "g": groups}).dropna()
    per = d.groupby("g")[["a", "b"]].apply(
        lambda x: x["a"].corr(x["b"]) if len(x) > min_n else np.nan
    )
    return per.mean()


ret = pd.to_numeric(exposures[config.RETURN_COL], errors="coerce")
lagged = exposures.sort_values(["id", "eom"]).groupby("id")[style_cols].shift(1)

timing = pd.DataFrame({
    "contemporaneous": [xs_corr(exposures[c], ret, exposures["eom"]) for c in style_cols],
    "lagged 1 month": [xs_corr(lagged[c], ret, exposures["eom"]) for c in style_cols],
}, index=[c[2:] for c in style_cols])
timing["shift"] = timing["lagged 1 month"] - timing["contemporaneous"]
display(timing.round(3))

print("corr(z_be_me, same-month return) is the mechanical channel:")
print("  book/market rises exactly when price falls, within the same month.")
print(f"\n{'⚠️ ' if timing['contemporaneous'].abs().max() > 0.03 else '✅'} "
      f"largest contemporaneous |corr| = {timing['contemporaneous'].abs().max():.3f}, "
      f"lagged = {timing['lagged 1 month'].abs().max():.3f}")

---
## Stage 3 — Fit every cross-section

`run_panel` does the lag, the winsorization and the constrained WLS. The
constraint is imposed by **substitution** — writing $f = U\theta$ for a
$K \times (K-1)$ basis $U$ that spans the constrained subspace — so it holds to
machine precision rather than approximately. A penalty would only satisfy it
approximately; dropping a column would silently re-baseline every industry
against the dropped one.

Regression weights are $\sqrt{\text{cap}}$. That is also what handles
cross-sectional heteroskedasticity, which is why Newey-West in 03 and 04 is
purely about serial correlation.

In [ ]:
F, U, diag, reg_panel = factor_returns.run_panel(exposures, style_cols, dummy_cols)

style_names = [c[2:] for c in style_cols]
core = [factor_returns.MARKET_FACTOR] + style_names
F.head()

### Verify the fit before believing any of it

Five properties the rest of the model depends on. The constraint check is the
important one — it is the whole reason this is not a one-line `lstsq`.

In [ ]:
scale = (diag["constraint_residual"].abs().max())
checks = {
    "every cross-section is full rank": bool(diag["full_rank"].all()),
    "cap-weighted industry constraint holds": scale < 1e-8,
    "no month lost to too-few-assets": len(F) == exposures["eom"].nunique() - 1,
    "no empty industry in any month": int(diag["n_empty_industries"].sum()) == 0,
    "R² is in a sane band": bool((diag["r2"] > 0).all() and (diag["r2"] < 0.95).all()),
    "every factor estimated in every month": bool(F.notna().all().all()),
}
for name, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {name}")

print(f"\nmax |constraint residual|   {scale:.2e}")
print(f"design condition number     median {diag['cond'].median():.1f}, max {diag['cond'].max():.1f}")
print(f"assets per cross-section    median {diag['n_assets'].median():.0f}, "
      f"min {diag['n_assets'].min():.0f}")
print(f"residual df                 median {diag['resid_df'].median():.0f}, "
      f"min {diag['resid_df'].min():.0f}")
display(diag[["n_assets", "resid_df", "r2", "cond", "exposures_filled"]].describe().round(3))

In [ ]:
# Fit quality through time. The universe is dynamic -- names list and delist --
# so asset counts move with the market rather than only falling as the sample's did.
fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))

ax[0].plot(diag["eom"], diag["r2"], color=INK, lw=1.2)
ax[0].axhline(diag["r2"].median(), color=MUTED, ls="--", lw=1)
ax[0].annotate(f"median {diag['r2'].median():.2f}", (diag["eom"].iloc[5], diag["r2"].median()),
               textcoords="offset points", xytext=(0, 6), color=MUTED, fontsize=8)
ax[0].set(title="Cross-sectional R²", ylabel="R²", ylim=(0, 1))

ax[1].plot(diag["eom"], diag["n_assets"], color=INK, lw=1.2)
ax[1].axhline(n_factors, color=ACCENT, ls="--", lw=1)
ax[1].annotate(f"K = {n_factors} parameters", (diag["eom"].iloc[5], n_factors),
               textcoords="offset points", xytext=(0, 6), color=ACCENT, fontsize=8)
ax[1].set(title="Assets per cross-section", ylabel="assets", ylim=(0, None))

for a in ax:
    a.grid(alpha=.25, lw=.6)
    a.set_axisbelow(True)
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

---
## Stage 4 — The factor returns

Annualized from monthly: mean × 12, volatility × √12. The `t` column is the
simple time-series t-statistic on the mean, which is a fair description of the
*sample* but is not a claim about out-of-sample premia. Unlike the 220-name
sample there is no survival screen here, so delisted firms are in.

In [ ]:
def annualize(fr):
    n = fr.notna().sum()
    return pd.DataFrame({
        "mean_ann_%": fr.mean() * 12 * 100,
        "vol_ann_%": fr.std() * np.sqrt(12) * 100,
        "t": fr.mean() / (fr.std() / np.sqrt(n)),
        "n": n,
    })


summary = annualize(F)
summary["kind"] = np.where(summary.index == factor_returns.MARKET_FACTOR, "market",
                    np.where(summary.index.str.startswith("ind_"), "industry", "style"))
display(summary.round(2))

print("Sanity anchors — these are the numbers that say the model is calibrated:")
print(f"  market factor      {summary.loc['market', 'mean_ann_%']:.1f}% / yr at "
      f"{summary.loc['market', 'vol_ann_%']:.1f}% vol   (a US equity market)")
sty = summary[summary["kind"] == "style"]
print(f"  style vols         {sty['vol_ann_%'].min():.1f}–{sty['vol_ann_%'].max():.1f}% / yr"
      f"   (USE4 styles run ~2–6%)")
ind = summary[summary["kind"] == "industry"]
print(f"  industry vols      {ind['vol_ann_%'].min():.1f}–{ind['vol_ann_%'].max():.1f}% / yr")

In [ ]:
# Cumulative style factor returns. Direct-labelled at the right edge as well as
# legended, so identity never rests on colour alone.
cum = F[style_names].cumsum()

fig, ax = plt.subplots(figsize=(13, 4.2))
for name, color in zip(style_names, STYLE_COLORS):
    ax.plot(cum.index, cum[name], color=color, lw=2, label=name)

# Direct labels, nudged apart where two series finish close together. Without
# this "beta" and "investment" land on top of each other and the direct label --
# the thing carrying identity for the lower-contrast hues -- stops being legible.
ends = cum.iloc[-1].sort_values()
min_gap = (cum.values.max() - cum.values.min()) * 0.045
placed = {}
last = -np.inf
for name in ends.index:
    y = max(ends[name], last + min_gap)
    placed[name] = y
    last = y

for name, color in zip(style_names, STYLE_COLORS):
    ax.annotate(f" {name}", (cum.index[-1], placed[name]), color=color,
                fontsize=9, va="center", fontweight="medium")

ax.axhline(0, color=MUTED, lw=1)
ax.set(title="Cumulative pure style factor returns (sum of monthly, not compounded)",
       ylabel="cumulative return")
ax.legend(ncol=6, frameon=False, loc="upper left", fontsize=9)
ax.grid(alpha=.25, lw=.6)
ax.set_axisbelow(True)
ax.margins(x=.06)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Industries as small multiples rather than 11 lines on one axis -- past about
# eight series a single panel stops being readable whatever the palette.
ind_cum = F[dummy_cols].cumsum()
fig, axes = plt.subplots(3, 4, figsize=(13, 6), sharex=True, sharey=True)
lo, hi = ind_cum.min().min(), ind_cum.max().max()

for ax, col in zip(axes.ravel(), dummy_cols):
    ax.plot(ind_cum.index, ind_cum[col], color=INK, lw=1.5)
    ax.axhline(0, color=MUTED, lw=.8)
    ax.set_title(col, fontsize=9)
    ax.grid(alpha=.25, lw=.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
for ax in axes.ravel()[len(dummy_cols):]:
    ax.set_visible(False)

fig.suptitle("Cumulative industry factor returns, relative to the market", fontsize=11)
fig.supylabel("cumulative return", fontsize=9)
plt.tight_layout()
plt.show()

---
## Stage 5 — A first look at `F`

This is not the covariance the model will ship — 03 adds Newey-West, exponential
decay, the Volatility Regime Adjustment and the eigenfactor adjustment, all on a
rolling window. This is the raw sample covariance of the fitted factor returns,
and its job here is to show whether the regression produced something a
covariance estimator can work with at all.

What to look for: style correlations should be **small**. These are *pure* factor
returns — each one is already purged of the others and of industry — so they
should not look like long-short strategy returns, which are much more correlated.
A large off-diagonal here means the regression is trading one factor off against
another, not that the factors are economically related.

In [ ]:
corr = F[core].corr()

fig, ax = plt.subplots(figsize=(7, 5.6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(core)), core, rotation=45, ha="right")
ax.set_yticks(range(len(core)), core)
# Every cell carries its number, so the reading never depends on the colour.
for i in range(len(core)):
    for j in range(len(core)):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(v) > .55 else "#1a1a19")
plt.colorbar(im, shrink=.8, label="correlation")
ax.set_title("Factor RETURN correlations — market + styles")
plt.tight_layout()
plt.show()

off = corr.where(~np.eye(len(corr), dtype=bool)).abs()
worst = off.stack().idxmax()
print(f"largest off-diagonal |correlation| = {off.max().max():.3f}  ({worst[0]} ↔ {worst[1]})")
print("\nmarket ↔ beta is expected and correct: Beta is deliberately NOT")
print("orthogonalized against the market in USE4, so the beta factor return")
print("carries market direction by design.")

In [ ]:
# Conditioning of the full K x K sample covariance. 03 replaces this, but if the
# raw version is already indefinite there is a problem upstream of the estimator.
cov = F.cov()
ev = np.linalg.eigvalsh(cov.values)

print(f"T = {len(F)} months, K = {cov.shape[0]} factors  →  T/K = {len(F) / cov.shape[0]:.1f}")
print(f"eigenvalues   min {ev.min():.3e}   max {ev.max():.3e}   condition {ev.max() / ev.min():.1f}")
print(f"{'✅' if (ev > 0).all() else '❌'} positive definite ({int((ev <= 0).sum())} non-positive eigenvalues)")
print("\n⚠️  T/K here is the FULL-SAMPLE ratio. On the 60-month rolling window 03")
print("    will use, T/K ≈ 3.3 — where eigenvalue bias is severe and the")
print("    eigenfactor adjustment stops being a refinement.")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(range(1, len(ev) + 1), ev[::-1], color=INK, width=.65)
ax.set(title="Eigenvalues of the sample factor covariance", xlabel="eigenvalue rank",
       ylabel="variance", yscale="log")
ax.grid(alpha=.25, lw=.6, axis="y")
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

### Which fix is doing the work

Both knobs off, both on, and each alone. Worth running rather than assuming,
because the intuitive attribution is **wrong**: the outlier looks like the
disease and is actually a symptom.

On the sample the culprit was SHARPLINK INC: +2225% in 2025-05, with its cap going
from \$2.0M to \$4,575M during that same month, so a contemporaneous fit weighted
it by the *post-spike* cap. Lagging took the weight from the pre-spike cap and
the leverage disappeared without any trimming. The largest-leverage rows in this
universe are listed under the table.

Watch the second row especially. Winsorizing *without* fixing the timing makes
the look-ahead look **more** convincing, not less — it strips the noise while
leaving the bias intact, so the spurious value premium's t-statistic goes from
−4.7 to −12.4. Fixing the symptom would have produced a confidently wrong model.

In [ ]:
grid = []
for lag in (False, True):
    for win in (None, config.WINSOR_RETURN_SIGMA):
        Fg, _, _, _ = factor_returns.run_panel(
            exposures, style_cols, dummy_cols,
            lag_exposures=lag, winsor_return_sigma=win, verbose=False,
        )
        cg = Fg[style_names].corr()
        off_diag = ~np.eye(len(style_names), dtype=bool)
        grid.append({
            "lag": "on" if lag else "off",
            "winsor": "off" if win is None else f"{win:g}σ",
            "corr(value,momentum)": cg.loc["value", "momentum"],
            "max |style corr|": cg.where(off_diag).abs().max().max(),
            "max style vol %/yr": Fg[style_names].std().max() * np.sqrt(12) * 100,
            "value %/yr": Fg["value"].mean() * 12 * 100,
            "value t": Fg["value"].mean() / (Fg["value"].std() / np.sqrt(len(Fg))),
            "market vol %/yr": Fg["market"].std() * np.sqrt(12) * 100,
        })

display(pd.DataFrame(grid).round(3))

print("On the sample the lag was the primary fix: it corrected the SIGN of the value")
print("premium (−18.1%/yr → +2.3%/yr). The rows above should show the same ordering here.")
# The observations with the most leverage in a contemporaneous fit: extreme return
# AND a large end-of-month cap. These are what the lag defuses.
ret_ = pd.to_numeric(exposures[config.RETURN_COL], errors="coerce")
lev = exposures.assign(abs_ret=ret_.abs(),
                       leverage=ret_.abs() * np.sqrt(pd.to_numeric(exposures[config.CAP_COL], errors="coerce")))
print("\nHighest-leverage observations in the unlagged fit:")
display(lev.nlargest(8, "leverage")[["eom", "conm", config.RETURN_COL, config.CAP_COL]])

---
## Stage 6 — The specific returns

`u(i,t)` is the input to `Δ` in 04. Two properties matter now.

**Orthogonality.** The residuals are weight-orthogonal to every column of the
design by construction, so a non-zero value here would mean the solver is wrong.

**Dispersion through time.** The cross-sectional spread of `u` is the raw
material for specific risk, and it is strongly time-varying — which is exactly
why 04 needs exponential decay and a volatility regime adjustment rather than a
flat historical mean.

In [ ]:
u_stats = U.groupby("eom")["specific_return"].agg(["count", "mean", "std"])
u_stats["ann_vol_%"] = u_stats["std"] * np.sqrt(12) * 100

print(f"specific returns: {len(U):,} rows, {U['id'].nunique()} securities")
print(f"cross-sectional σ(u): median {u_stats['std'].median():.4f}/month "
      f"({u_stats['ann_vol_%'].median():.1f}%/yr annualized)")
print(f"   range {u_stats['ann_vol_%'].min():.1f}% – {u_stats['ann_vol_%'].max():.1f}%/yr"
      f"   — a {u_stats['ann_vol_%'].max() / u_stats['ann_vol_%'].min():.1f}× swing")
print(f"\n{'✅' if U['specific_return'].abs().max() < 2 else '⚠️ '} "
      f"max |u| = {U['specific_return'].abs().max():.3f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
ax[0].plot(u_stats.index, u_stats["ann_vol_%"], color=INK, lw=1.2)
ax[0].set(title="Cross-sectional dispersion of specific returns",
          ylabel="annualized σ(u), %")
ax[1].hist(U["specific_return"], bins=120, color=INK)
ax[1].set(title="Distribution of specific returns", xlabel="u", ylabel="observations",
          yscale="log")
for a in ax:
    a.grid(alpha=.25, lw=.6)
    a.set_axisbelow(True)
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Serial correlation in u is what Newey-West exists to handle in 04. Measure it
# now so that choice is evidence-based rather than inherited.
ac = (U.sort_values(["id", "eom"])
        .groupby("id")["specific_return"]
        .apply(lambda s: s.autocorr(lag=1) if s.notna().sum() > 24 else np.nan)
        .dropna())
print(f"lag-1 autocorrelation of u, across {len(ac)} securities:")
print(f"   mean {ac.mean():+.3f}   median {ac.median():+.3f}   "
      f"[p10 {ac.quantile(.1):+.3f}, p90 {ac.quantile(.9):+.3f}]")
print(f"   share negative: {(ac < 0).mean():.0%}")
print(f"\nrule-of-thumb Newey-West lag for T = {len(F)}: "
      f"L ≈ 4(T/100)^(2/9) = {4 * (len(F) / 100) ** (2 / 9):.1f}")

---
## Stage 7 — Write the artifacts

Three tables. Factor returns go out **long** — `(eom, factor, factor_return)` —
because 03 needs to pivot them by forecast date and a long table survives adding
or removing a factor without a schema change. Specific returns are `(id, eom)`,
the grain `Δ` is estimated on.

In [ ]:
f_long = factor_returns.to_long(F)
db.write_table(f_long, config.TBL_FACTOR_RETURNS, engine, unique_on=["eom", "factor"])

u_out = U[["id", "eom", "specific_return", "reg_weight"]].copy()
db.write_table(u_out, config.TBL_SPECIFIC_RETURNS, engine, unique_on=["id", "eom"])

db.write_table(diag, config.TBL_REGRESSION_DIAG, engine, unique_on=["eom"])

# The regression panel: every row the fit actually saw, carrying BOTH dates.
# Stored because ryan_<profile>_exposures is keyed (id, eom) meaning "the exposures
# observed at eom" — true, but a trap for anything that reads it. Regressing ITS
# ret_exc on ITS OWN f_* columns is the contemporaneous fit, and it reports a
# value factor return of −18.1%/yr at t = −4.7 with every diagnostic looking
# healthy. Here the alignment is legible in the data instead of implicit in code.
panel_cols = (["id", "eom", "exposure_eom", config.RETURN_COL, "ret_exc_raw",
               config.CAP_COL] + style_cols + dummy_cols)
db.write_table(reg_panel[[c for c in panel_cols if c in reg_panel.columns]],
               config.TBL_REGRESSION_PANEL, engine, unique_on=["id", "eom"])

display(f_long.head())
print(f"\nfactor returns   {len(f_long):,} rows  ({F.shape[1]} factors × {len(F)} months)")
print(f"specific returns {len(u_out):,} rows")
print(f"diagnostics      {len(diag):,} rows")
print(f"regression panel {len(reg_panel):,} rows")

gap = (reg_panel["eom"].dt.to_period("M")
       - reg_panel["exposure_eom"].dt.to_period("M")).apply(lambda x: x.n)
print(f"{'✅' if (gap == 1).all() else '❌'} exposure_eom is exactly one month "
      f"before eom in every row of the stored panel")

---
## What 03 does next

`F(t)`, the rolling covariance of these factor returns. The sample showed that at
T/K ≈ 3.3 the eigenfactor adjustment is a correctness requirement and that
Newey-West made F worse. 03 re-runs both tests on this universe instead of
assuming they carry over.